#### Model Training and Evaluation Workflow

In this section, we will go through the entire process of training and evaluating a machine learning model for predicting NBA game outcomes. Below is an overview of the steps taken:

##### 1. **Removing Any Existing Model File**
Before training a new model, we ensure that any previously saved model file (`deepshot.pkl`) is deleted to prevent conflicts when saving the updated version.

##### 2. **Loading the Dataset**
We use `pandas` to load the dataset from a CSV file. A loading spinner is displayed using `rich` to provide visual feedback while the data is being read.

##### 3. **Preprocessing the Data**
- We drop unnecessary columns, such as `home_team` and `away_team`, since they are not used as input features for training.
- Any additional irrelevant statistical features (if specified) are also removed to optimize model performance.

##### 4. **Defining Features and Target Variable**
- The feature matrix `X` consists of all columns except the `winning_team` column.
- The target variable `y` is set as `winning_team`, which is already encoded as `0` (home win) and `1` (away win).

##### 5. **Splitting the Dataset**
The dataset is split into training and testing sets using an 80-20 ratio to evaluate model performance effectively.

##### 6. **Training the Model**
- A `XGBoostClassifier` is initialized with specific hyperparameters to improve model performance.
- A progress spinner is displayed while the model is being trained.

##### 7. **Evaluating the Model**
- Predictions are made on the test set.
- The model's accuracy is calculated and displayed using `rich` for better readability.
- A classification report is printed to show precision, recall, and F1-score for each class.

##### 8. **Saving the Model**
Finally, the trained model is saved as `deepshot.pkl` using `joblib` for future use.

This structured approach ensures that the model is trained, evaluated, and saved efficiently, with informative feedback at each step.



In [1]:
# Importing libraries
from pathlib import Path
import pandas as pd
from xgboost import XGBClassifier
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score, 
    log_loss, 
    brier_score_loss, 
    classification_report)
import joblib

# Removing the already existing model to overwrite it with the new one
Path("deepshot.pkl").unlink(missing_ok=True)

# Start loading data
print("Loading Data...")

# Load CSV Data
df: pd.DataFrame = pd.read_csv("../data/csv/dataset.csv")
print("Data Loaded Successfully!")

# Convert date column for time-based split
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Drop columns that are not used for training (just home_team and away_team)
df: pd.DataFrame = df.drop(["home_team", "away_team"], axis=1)

# Drop irrelevant stats columns
stats_to_drop: list[str] = list()
for stat in stats_to_drop:
    df: pd.DataFrame = df.drop([f"home_{stat}", f"away_{stat}"], axis=1)

# Train on historical seasons (up to 2023-24), test on 2024-25 season
print("Splitting the dataset into training and testing sets...")
split_date: str = "2024-10-01"
train_df: pd.DataFrame = df[df["date"] < split_date].reset_index(drop=True)
test_df: pd.DataFrame = df[df["date"] >= split_date].reset_index(drop=True)
print(f"Training games: {len(train_df)}, Test games: {len(test_df)}")

# Features and target
X_train: pd.DataFrame = train_df.drop(["winning_team", "date"], axis=1)
y_train: pd.Series = train_df["winning_team"]
X_test: pd.DataFrame = test_df.drop(["winning_team", "date"], axis=1)
y_test: pd.Series = test_df["winning_team"]

imbalance_ratio: float = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Class imbalance ratio (Home:Away) ≈ {imbalance_ratio:.2f}")

# Train the XGBoost model
xgb: XGBClassifier = XGBClassifier(
    colsample_bytree=0.872,
    gamma=3.45,
    learning_rate=0.01,
    max_depth=3,
    min_child_weight=1,
    n_estimators=330,
    objective="binary:logistic",
    reg_alpha=4.58,
    reg_lambda=4.23,
    subsample=0.7,
    scale_pos_weight=0.95,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

print("Training XGBoost model...")
xgb.fit(X_train, y_train)
print("Model training complete!")

# Evaluate Model
y_pred: np.ndarray = xgb.predict(X_test)
y_pred_proba: np.ndarray = xgb.predict_proba(X_test)[:, 1]

# Checking various metrics to not use only one indicator
accuracy: float = accuracy_score(y_test, y_pred)
roc_auc: float = roc_auc_score(y_test, y_pred_proba)
logloss: float = log_loss(y_test, y_pred_proba)
brier: float = brier_score_loss(y_test, y_pred_proba)

print(f"Accuracy: {accuracy*100:.2f}%")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Log Loss: {logloss:.4f}")
print(f"Brier Score: {brier:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Home Win", "Away Win"]))

# Probability calibration analysis
print("\nCalibration by predicted probability bins:")
prob_bins: list[float] = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

for i in range(len(prob_bins) - 1):
    mask: np.ndarray = (y_pred_proba >= prob_bins[i]) & (y_pred_proba < prob_bins[i + 1])
    if mask.sum() > 0:
        actual_rate: np.float64 = y_test[mask].mean()
        print(f"Predicted {prob_bins[i]:.1f}-{prob_bins[i+1]:.1f}: Actual away win rate = {actual_rate:.2%} (n={mask.sum()})")

# Saving the final calibrated model
joblib.dump(xgb, "deepshot.pkl")
print("\nCalibrated model saved as 'deepshot.pkl'")
print("\nPipeline completed successfully!")

Loading Data...
Data Loaded Successfully!
Splitting the dataset into training and testing sets...
Training games: 26632, Test games: 1231
Class imbalance ratio (Home:Away) ≈ 1.45
Training XGBoost model...
Model training complete!
Accuracy: 64.99%
ROC-AUC: 0.7174
Log Loss: 0.6189
Brier Score: 0.2151

Classification Report:
              precision    recall  f1-score   support

    Home Win       0.64      0.81      0.72       669
    Away Win       0.67      0.46      0.54       562

    accuracy                           0.65      1231
   macro avg       0.66      0.63      0.63      1231
weighted avg       0.65      0.65      0.64      1231


Calibration by predicted probability bins:
Predicted 0.0-0.5: Actual away win rate = 35.97% (n=848)
Predicted 0.5-0.6: Actual away win rate = 57.45% (n=188)
Predicted 0.6-0.7: Actual away win rate = 73.24% (n=142)
Predicted 0.7-0.8: Actual away win rate = 84.91% (n=53)

Calibrated model saved as 'deepshot.pkl'

Pipeline completed successfully!


In [2]:
# Importing libraries
import joblib
import pandas
from xgboost import XGBClassifier
import numpy as np

# Loading the model
xgb: XGBClassifier = joblib.load("deepshot.pkl")

# Load CSV Data
df: pandas.DataFrame = pandas.read_csv("../data/csv/dataset.csv")

# Drop columns that are not used for training (just home_team and away_team)
df: pandas.DataFrame = df.drop(["date", "home_team", "away_team"], axis=1)

# Drop irrelevant stats columns
stats_to_drop: list[str] = []
for stat in stats_to_drop:
    df: pandas.DataFrame = df.drop([f"home_{stat}", f"away_{stat}"], axis=1)

# Define features (X) and target (y)
X: pandas.DataFrame = df.drop(
    "winning_team", axis=1
)  # Features (all columns except 'winning_team')
y: pandas.Series = df["winning_team"]  # Target variable (already 0 or 1)

# Set options to display all columns
pandas.set_option("display.max_columns", None)  # No column truncation
pandas.set_option("display.width", None)  # Automatically adjust width
pandas.set_option("display.max_rows", None)  # Show all rows, if necessary

# Now print the feature importance DataFrame
print("\nFeature Importances:")
feature_importances: np.ndarray = xgb.feature_importances_
features: pandas.core.indexes.base.Index = X.columns  # Get the feature names

# Create a DataFrame to visualize feature importance
importance_df: pandas.DataFrame = pandas.DataFrame({"Feature": features, "Importance": feature_importances})

# Sort by importance in descending order
importance_df: pandas.DataFrame = importance_df.sort_values(by="Importance", ascending=False)

# Print the DataFrame with feature importances
print(importance_df)

# Reset Pandas display options to default (optional)
pandas.reset_option("display.max_columns")
pandas.reset_option("display.width")
pandas.reset_option("display.max_rows")


Feature Importances:
           Feature  Importance
37        home_elo    0.085394
75        away_elo    0.069723
22       home_drtg    0.043168
59       away_ortg    0.041623
69    away_efg_pct    0.039277
21       home_ortg    0.033488
64         away_ts    0.029715
60       away_drtg    0.023258
38        away_pts    0.021640
63      away_3ptar    0.020566
73    away_ast_tov    0.018439
3      home_fg_pct    0.016339
46       away_fg2a    0.015610
27    home_trb_pct    0.015471
47    away_fg2_pct    0.015030
54        away_ast    0.014383
74  away_ast_ratio    0.013103
26         home_ts    0.012591
66    away_ast_pct    0.012569
36  home_ast_ratio    0.012407
42        away_fg3    0.011762
70    away_tov_pct    0.011355
5        home_fg3a    0.011293
43       away_fg3a    0.011013
13        home_orb    0.010919
34    home_ft_rate    0.010782
35    home_ast_tov    0.010678
19        home_tov    0.010630
57        away_tov    0.010389
56        away_blk    0.010223
41     away_fg_pc

#### Model Loading and Feature Importance Analysis

In this section, we load a pre-trained machine learning model and prepare the dataset for evaluation. The steps are as follows:

1. **Load the Model:** We use `joblib` to load a previously trained model (`deepshot.pkl`).
2. **Load and Preprocess Data:** The dataset is read from a CSV file. Since some columns like `home_team` and `away_team` are not used for training, we drop them.
3. **Define Features and Target Variable:**
   - `X`: Contains all feature columns except the target variable (`winning_team`).
   - `y`: The target variable, which indicates whether the home team won (0) or the away team won (1).
4. **Split Data for Testing:** We split the dataset into training and testing sets using an 80/20 ratio.
5. **Feature Importance Analysis with SHAP:**
   - A sample of 250 test instances is selected.
   - SHAP (SHapley Additive exPlanations) values are computed using a TreeExplainer.
   - The feature importance is visualized using a SHAP summary plot to understand how different features influence the model’s predictions.

This analysis helps us interpret the model’s decision-making process and identify the most influential factors in predicting game outcomes.

In [3]:
# Importing libraries
import joblib
from xgboost import XGBClassifier
import pandas
from sklearn.model_selection import train_test_split
import shap


# Loading the model
xgb: XGBClassifier = joblib.load("deepshot.pkl")

# Load CSV Data
df: pandas.DataFrame = pandas.read_csv("../data/csv/dataset.csv")

# Drop columns that are not used for training (just home_team and away_team)
df: pandas.DataFrame = df.drop(["date", "home_team", "away_team"], axis=1)

# Define features (X) and target (y)
X: pandas.DataFrame = df.drop(
    "winning_team", axis=1
)  # Features (all columns except 'winning_team')
y: pandas.Series = df["winning_team"]  # Target variable (already 0 or 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Analyzing feature importance with SHAP values and chart
sample: pandas.core.frame.DataFrame = X_test.sample(1000, random_state=42)
explainer: shap.explainers._tree.TreeExplainer = shap.TreeExplainer(xgb, sample)
shap_values: shap._explanation.Explanation = explainer(sample)
shap.summary_plot(shap_values, sample)

ModuleNotFoundError: No module named 'shap'

In [1]:
pip install shap


  Using cached shap-0.51.0-cp312-cp312-win_amd64.whl.metadata (26 kB)
  Using cached numpy-2.4.2-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
Using cached shap-0.51.0-cp312-cp312-win_amd64.whl (556 kB)
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   -- ------------------------------------- 1.0/15.6 MB 6.3 MB/s eta 0:00:03
   -- ------------------------------------- 1.0/15.6 MB 6.3 MB/s eta 0:00:03
   ------ --------------------------------- 2.4/15.6 MB 3.7 MB/s eta 0:00:04
   -------- ------------------------------- 3.1/15.6 MB 3.8 MB/s eta 0:00:04
   ---------- ----------------------------- 3.9/15.6 MB 3.9 MB/s eta 0:00:04
   ------------ --------------------------- 4.7/15.6 MB 3.9 MB/s eta 0:00:03
   -------------- ------------------------- 5.8/15.6 MB 3.8 MB/s eta 0:00:03
   ---------------- ----------------------- 6.6/15.6 MB 3.8 MB/s eta 0:00:03

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.0.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
